# README
This notebook is used to obtain the embeddings of the domain-knowledge prompt for one dataset

In [1]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib as mpl
import pickle
import transformers
from matplotlib.lines import Line2D
import seaborn as sns
import json
import torch
from Prompts.Mapping_helper import Mapping_helper
from transformers import GPT2Config, GPT2Tokenizer, GPT2Model, AutoTokenizer, AutoModel, AutoConfig, Phi3Config

In [2]:
def set_ax_linewidth(ax, bw=1.5):
    ax.spines['bottom'].set_linewidth(bw)
    ax.spines['left'].set_linewidth(bw)
    ax.spines['top'].set_linewidth(bw)
    ax.spines['right'].set_linewidth(bw)

def set_ax_font_size(ax, fontsize=10):
    ax.tick_params(axis='y',
                 labelsize=fontsize # y轴字体大小设置
                  ) 
    ax.tick_params(axis='x',
                 labelsize=fontsize # x轴字体大小设置
                  ) 

def set_draft(the_plt, other_ax=''):
    ax = the_plt.gca()
    ax.axes.xaxis.set_ticklabels([])
    ax.axes.yaxis.set_ticklabels([])
    plt.xlabel('')
    plt.ylabel('')
    if other_ax:
        other_ax.axes.xaxis.set_ticklabels([])
        other_ax.axes.yaxis.set_ticklabels([])
        other_ax.set_ylabel('')
        other_ax.set_xlabel('')

def set_draft_fig(fig):
    for ax in fig.axes:
        ax.axes.xaxis.set_ticklabels([])
        ax.axes.yaxis.set_ticklabels([])
        ax.set_ylabel('')
        ax.set_xlabel('')

In [3]:
target_dataset = 'total' 
cell_names = json.load(open('data_provider/split_json/total_split_2021.json'))['train'] + json.load(open('data_provider/split_json/total_split_2021.json'))['val'] + json.load(open('data_provider/split_json/total_split_2021.json'))['test']
cell_names = list(set(cell_names))

## Tokenize and embedding

In [4]:
def create_causal_mask(B, seq_len):
    '''
    return:
        casual mask: [B, L, L]. 0 indicates masked.
    '''
    # Create a lower triangular matrix of shape (seq_len, seq_len)
    mask = torch.tril(torch.ones(seq_len, seq_len))  # (L, L)
    mask = mask.unsqueeze(0).expand(B, -1, -1)
    return mask

def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

In [5]:
# loader the tokenizer and model

# '/data/LLMs/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659'
# '/data/LLMs/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418'
# '/data/LLMs/models--Qwen--Qwen3-Embedding-0.6B/snapshots/744169034862c8eec56628663995004342e4e449'
# 'Qwen/Qwen3-Embedding-0.6B'
LLM_path = '/data/LLMs/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418'
llama_config = AutoConfig.from_pretrained(LLM_path)
if 'Qwen3-Embedding-0.6B' in LLM_path:
    language_model = AutoModel.from_pretrained(
                LLM_path
            ).cuda()
else:
    language_model = AutoModel.from_pretrained(
                LLM_path,
                # 'huggyllama/llama-7b',
                trust_remote_code=True,
                local_files_only=True,
                config=llama_config,
                load_in_4bit=True
            )
if 'Llama' in LLM_path:
    tokenizer = AutoTokenizer.from_pretrained(
                    LLM_path,
                    # 'huggyllama/llama-7b',
                    trust_remote_code=True,
                    local_files_only=True, 
                    pad_token='<|endoftext|>'
                )
    tokenizer.padding_side = 'right' # set the padding side
else:
    tokenizer = AutoTokenizer.from_pretrained(LLM_path, padding_side='left')


In [6]:
def get_features_from_cellNames(cell_names):
    cellName_prompt = {}
    for cell_name in cell_names:
        # bg_prompt = (
        #             f"Task description: You are an expert in predicting battery cycle life. " 
        #             f"The cycle life is the number of cycles until the battery's discharge capacity reaches 80% of its nominal capacity. "
        #             f"The discharge capacity is calculated under the described operating condition. "
        #             f"Please directly output the cycle life of the battery based on the provided data. "
        #             )
        if 'CALB' in cell_name:
            bg_prompt = (
                        f"Task description: " 
                        f"The end of life of a battery is the number of charge-discharge cycles until the battery's discharge capacity reaches 90% of its nominal capacity. "
                        f"The discharge capacity is calculated under the described operating condition. "
                        f"The state of the health (SOH) is computed by the ratio of the remaining capacity to the initial capacity. "
                        f"The target is the SOH degradation trajecotry until the end of life of the battery."
                        f"Please directly output the target of the battery based on the provided data. "
                        )
        else:
            bg_prompt = (
                        f"Task description: " 
                        f"The end of life of a battery is the number of charge-discharge cycles until the battery's discharge capacity reaches 80% of its nominal capacity. "
                        f"The discharge capacity is calculated under the described operating condition. "
                        f"The state of the health (SOH) is computed by the ratio of the remaining capacity to the nominal capacity. "
                        f"The target is the SOH degradation trajecotry until the end of life of the battery."
                        f"Please directly output the target of the battery based on the provided data. "
                        )
        
        cell_name = cell_name.split('.pkl')[0]
        helper = Mapping_helper(prompt_type='PROTOCOL', cell_name=cell_name)
        tmp_prompt = bg_prompt + helper.do_mapping()
        if 'Llama' in LLM_path:
            messages = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": tmp_prompt}
            ]

            tmp_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            res = tokenizer(tmp_prompt, return_tensors="pt")
            input_ids, attention_mask = res['input_ids'][:,1:], res['attention_mask'][:,1:]
            llama_enc_out = language_model.get_input_embeddings()(input_ids) # [1, L', d_llm]
            
            cache_position = torch.arange(
                    0, 0 + llama_enc_out.shape[1], device=llama_enc_out.device
                )
            position_ids = cache_position.unsqueeze(0)
            DLP_attention_mask = attention_mask.unsqueeze(1) # [B, 1, L]
            DLP_attention_mask = DLP_attention_mask.expand(-1, DLP_attention_mask.shape[-1], -1) # [B, L, L]
            DLP_attention_mask = DLP_attention_mask.unsqueeze(1) # [B, 1, L, L]
            
            casual_mask = create_causal_mask(1, llama_enc_out.shape[1])
            casual_mask = casual_mask.unsqueeze(1) # [B, 1, L, L]

            DLP_attention_mask = torch.where(casual_mask.to(DLP_attention_mask.device)==1, DLP_attention_mask, torch.zeros_like(DLP_attention_mask))
            DLP_attention_mask = DLP_attention_mask==1 # set True to allow attention to attend to

            hidden_states = language_model(inputs_embeds=llama_enc_out).last_hidden_state
            # hidden_states = llama_enc_out
            # for i, layer in enumerate(language_model.layers):
            #     res = layer(hidden_states=hidden_states, position_ids=position_ids, attention_mask=DLP_attention_mask, cache_position=cache_position)
            #     hidden_states = res[0]

            features = hidden_states[:,-1,:].detach().cpu().numpy().reshape(1, -1)
        elif 'Qwen3' in LLM_path:
            tmp_prompt = [get_detailed_instruct('classification', tmp_prompt)]
            res = tokenizer(
                tmp_prompt,
                padding=True,
                truncation=True,
                max_length=8192,
                return_tensors="pt",
            )
            res.to(language_model.device)
            outputs = language_model(**res)
            embeddings = last_token_pool(outputs.last_hidden_state, res['attention_mask'])
            features = embeddings.detach().cpu().numpy().reshape(1, -1)
        else:
            raise Exception(f'{LLM_path} is not supported here')

        
    
        cellName_prompt[cell_name+'.pkl'] = features
    return cellName_prompt

print(len(cell_names))
cellName_prompt = get_features_from_cellNames(cell_names)
print(f'The LLM embedding dim is {cellName_prompt[cell_names[0]].shape[1]}')

1129
The LLM embedding dim is 1024


## Save the results

In [7]:
## Export the domain-knowledge prompt embeddings of the samples
save_path = './data_provider/prompt_embeddings/'
print(len(cellName_prompt))
print(f'{save_path}Qwen3_{target_dataset}.pkl')
with open(f'{save_path}Qwen3_{target_dataset}.pkl', 'wb') as f:
    pickle.dump(cellName_prompt, f)

1129
./data_provider/prompt_embeddings/Qwen3_total.pkl
